**Calibração MACI (Agregação Multiplicativa)**

Calcula o resultado estatístico final

**Agregação Multiplicativa (Estado da Arte - MACI)**

Uma técnica recente (2024/2025) chamada Multi-LLM Adaptive Conformal Inference (MACI) propõe modelar a factualidade como o produto dos scores das sentenças ($s_{doc} = 1 - \prod (1 - s_i)$). Isso é mais robusto e evita que o sistema seja sensível demais a um único erro de estimativa de uma frase.4. Ajuste no Roteiro de ImplementaçãoPara o seu próximo script, vamos organizar os dados assim:Agrupar por id_bo: Calcular a média e o máximo dos scores de cada boletim.Reportar Factualidade Agregada: No artigo, você apresentará uma tabela com:% de Sentenças Factuais (Finitas).% de Relatórios Íntegros (Document-level).

In [3]:
import pandas as pd
import numpy as np

In [ ]:
# MÉTODO AVULSO PARA REMOVER AS TAGS <think>...</think> DO CAMPO relatorio_ia
import re
import pandas as pd

def remover_think_relatorio(csv_path: str, output_path: str = None) -> pd.DataFrame:
    # Carregar o CSV
    df = pd.read_csv(csv_path)
    
    if "relatorio_ia" not in df.columns:
        raise ValueError("CSV deve conter coluna 'relatorio_ia'.")

    # Regex para tirar as tags <think> e </think> e tudo entre elas
    pattern = re.compile(r"<think>.*?</think>", flags=re.DOTALL | re.IGNORECASE)

    df_limpo = df.copy()
    df_limpo["relatorio_ia"] = (
        df_limpo["relatorio_ia"]
        .astype(str)
        .apply(lambda x: pattern.sub("", x).strip())
    )

    # Salvar em arquivo se output_path foi fornecido
    if output_path:
        df_limpo.to_csv(output_path, index=False)
        print(f"✅ Arquivo salvo em: {output_path}")

    return df_limpo

# Exemplo de uso:
# df_limpo = remover_think_relatorio("THINK-dataset_com_relatorios.csv")
# remover_think_relatorio("THINK-dataset_com_relatorios.csv", output_path="dataset_com_relatorios1.csv")

✅ Arquivo salvo em: dataset_com_relatorios1.csv


,codigo_bo,narrativa_original,contexto_completo,relatorio_ia
0,1997,comunicante começou receber ligações mensagens...,CODIGO: 1997 | NATUREZA: OUTRAS FRAUDES | DATA...,**RELATÓRIO TÉCNICO RESUMIDO – BOLETIM DE OCOR...
1,1578,ligação mulher dizendo editora globo iriam ca...,CODIGO: 1578 | NATUREZA: ESTELIONATO | DATA: 2...,**RELATÓRIO TÉCNICO RESUMIDO – BOLETIM DE OCOR...
2,11656,narra comuncante numero celular conta gmail cl...,CODIGO: 11656 | NATUREZA: ESTELIONATO | DATA: ...,**RELATÓRIO TÉCNICO RESUMIDO – BOLETIM DE OCOR...
3,7703,tecnica enfermagem acordo comunicante mesma p...,CODIGO: 7703 | NATUREZA: ESTELIONATO | DATA: 2...,**RELATÓRIO TÉCNICO RESUMIDO – BOLETIM DE OCOR...
4,13564,redes sociais contas google hackeadas tereciro...,CODIGO: 13564 | NATUREZA: ESTELIONATO | DATA: ...,**RELATÓRIO TÉCNICO RESUMIDO – BOLETIM DE OCOR...
...,...,...,...,...
95,815,data 1724 verificou conta havia sido hackeada ...,CODIGO: 815 | NATUREZA: FURTO | DATA: 2024-02-...,**RELATÓRIO TÉCNICO RESUMIDO – BOLETIM DE OCOR...
96,13534,recebeu notificação protestos oficio 1º tabeli...,CODIGO: 13534 | NATUREZA: ESTELIONATO | DATA: ...,**RELATÓRIO TÉCNICO RESUMIDO – BOLETIM DE OCOR...
97,1597,comparece nesta central rosangela soares rel...,CODIGO: 1597 | NATUREZA: OUTRAS FRAUDES | DATA...,**RELATÓRIO TÉCNICO RESUMIDO – BOLETIM DE OCOR...
98,13070,recebeu mensagem suposto advogado identifica...,CODIGO: 13070 | NATUREZA: ESTELIONATO | DATA: ...,**RELATÓRIO TÉCNICO RESUMIDO – BOLETIM DE OCOR...


In [4]:
def calcular_maci_conformal(csv_auditado, alpha=0.05):
    df = pd.read_csv(csv_auditado)
    
    # 1. Agregação Multiplicativa (Frequência de Factualidade do Documento)
    # Probabilidade de ser fiel = 1 - s_i
    df['prob_fiel'] = 1 - df['non_conformity_score']
    
    # Produto das probabilidades por BO
    df_docs = df.groupby('id_bo')['prob_fiel'].prod().reset_index()
    
    # s_doc = 1 - Produto(prob_fiel)
    df_docs['s_doc'] = 1 - df_docs['prob_fiel']
    
    # 2. Particionamento
    df_calib = df_docs.sample(frac=0.5, random_state=42)
    df_test = df_docs.drop(df_calib.index)
    
    n = len(df_calib)
    scores_calib = df_calib['s_doc'].values
    
    # 3. Cálculo do Limiar Conformal q_hat
    # Fórmula rigorosa de Split Conformal: ceil((n+1)(1-alpha)) / (n+1)
    q_level = np.ceil((n + 1) * (1 - alpha)) / (n + 1)
    q_hat = pd.Series(scores_calib).quantile(q_level, interpolation='higher')
    
    # 4. Avaliação
    df_test['aprovado'] = df_test['s_doc'] <= q_hat
    taxa_aprovacao = df_test['aprovado'].mean()
    erro_residual = df_test[df_test['aprovado'] == True]['s_doc'].mean()

    print(f"--- RESULTADOS FINAIS (MACI) ---")
    print(f"Limiar de Risco Aceitável (q_hat): {q_hat:.4f}")
    print(f"Taxa de Relatórios Íntegros (Aprovados): {taxa_aprovacao*100:.2f}%")
    print(f"Risco de Alucinação nos Aprovados: {erro_residual*100:.4f}%")
    
    df_test.to_csv("relatorios_finalizados_conformal.csv", index=False)
    return q_hat

In [5]:
calcular_maci_conformal("sentencas_auditadas_votos.csv")

--- RESULTADOS FINAIS (MACI) ---
Limiar de Risco Aceitável (q_hat): 0.0000
Taxa de Relatórios Íntegros (Aprovados): 98.00%
Risco de Alucinação nos Aprovados: 0.0000%


np.float64(0.0)